# Unidade 3 — Guia de estudo

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/andrelfo85/Atividade_Goodreads/blob/master/unidade_03/00_guia_da_unidade.ipynb)

## Como transformar fontes em dados analisáveis?

**Problema orientador:** como preparar documentos e registros produzidos em
formatos diferentes sem perder a ligação com as fontes e decisões?

O protocolo da Unidade 2 definiu o que coletar. Agora construiremos uma
primeira base processável. “Limpar” não significa tornar os dados neutros:
significa aplicar regras explícitas, preservar originais, testar resultados e
documentar perdas. Rawson e Muñoz (2019) alertam que a ideia de limpeza pode
ocultar trabalho interpretativo e relações de autoridade.

![Fontes em papel, imagem e tabela atravessam camadas transparentes de transformação e produzem tabelas, relações e segmentos de texto sempre ligados aos originais preservados.](https://github.com/andrelfo85/Atividade_Goodreads/blob/master/unidade_03/imagens/00_abertura_conceitual.png?raw=1)

A ilustração antecipa o argumento central: transformar não rompe o vínculo
com a fonte. Cada produto derivado precisa manter um caminho de retorno às
entradas, regras e decisões que o produziram.

In [1]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_03'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = [('openpyxl', 'openpyxl>=3.1,<4'), ('pypdf', 'pypdf>=5,<6'), ('PIL', 'Pillow>=11,<12')]
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

Ambiente preparado em: /content/disciplina_computacao_aplicada_humanidades_digitais/unidade_03


O problema orientador define a exigência de preservar vínculos durante a
transformação. Para realizá-la, precisamos decompor o trabalho em etapas nas
quais cada produto possa ser verificado antes de alimentar o seguinte.

## Objetivos e percurso

| Notebook | Questão | Produto |
|---|---|---|
| 00 | Como trabalhar sem perder rastros? | Diagnóstico |
| 01 | Como ler formatos e extrair texto? | Inventário técnico |
| 02 | Como estruturar e corrigir com cautela? | Tabela intermediária |
| 03 | Como juntar e integrar sem criar casos? | Base integrada |
| 04 | A base está processável e auditável? | Primeira versão da base |

A unidade usa dados fictícios deliberadamente inconsistentes e um pequeno
extrato didático dos códigos de municípios do IBGE, acompanhado de
proveniência. Não há coleta pela internet durante a execução.

![Quatro etapas ligam leitura de formatos, estruturação, integração e empacotamento; os dados brutos permanecem preservados e uma seta de retorno representa auditoria e revisão.](https://github.com/andrelfo85/Atividade_Goodreads/blob/master/unidade_03/imagens/00_percurso_unidade.svg?raw=1)

O percurso é cumulativo: cada notebook produz uma representação que será
retomada pelo seguinte. Testes e limites podem exigir retorno a qualquer
decisão anterior.

## Como ler a pasta `dados`

A pasta está organizada como um fluxo de transformação. Os arquivos em
`brutos/` são entradas preservadas; `intermediarios/` guarda um resultado
verificável ainda sujeito a integração; e `derivados/` reúne produtos
reconstruíveis destinados à análise. Portanto, os arquivos não são exemplos
soltos: cada um permite examinar um problema metodológico específico.

### Entradas preservadas: `brutos/`

| Arquivo | O que representa | Problema ao qual se relaciona | Uso principal |
|---|---|---|---|
| `brutos/catalogo_messy.csv` | Catálogo fictício delimitado por ponto e vírgula, com datas, grafias, ausências e possível duplicata deliberadamente inconsistentes. | Importação parametrizada, diagnóstico de qualidade, padronização cautelosa e detecção de duplicatas. | Notebooks 01 e 02. |
| `brutos/catalogo_messy.xlsx` | O mesmo catálogo em uma planilha Excel. | Como formatos diferentes exigem leitores e parâmetros próprios, ainda que representem conteúdo semelhante. | Notebook 01. |
| `brutos/metadados.json` | Lista hierárquica que liga documentos a arquivos textuais e a zero ou vários temas. | Leitura de dados semiestruturados e representação de relações 1:N. | Notebooks 01 e 03. |
| `brutos/metadados.xml` | Pequeno recorte dos metadados em estrutura XML. | Diferenças de sintaxe, hierarquia e extração entre formatos semiestruturados. | Notebook 01. |
| `brutos/D001.txt` | Texto simples associado ao documento D001. | Leitura de texto não estruturado e manutenção do vínculo entre arquivo e identificador. | Notebooks 01 e 03. |
| `brutos/D002.txt` | Texto simples associado ao documento D002. | Integração de conteúdo textual com o catálogo sem copiar listas ou perder a chave documental. | Notebooks 01 e 03. |
| `brutos/documento_textual.pdf` | PDF sintético que já contém uma camada textual. | Distinguir extração de texto de PDF nascido digitalmente de OCR aplicado a imagem. | Notebook 01. |
| `brutos/pagina_digitalizada.png` | Imagem sintética limpa de uma linha tipográfica. | Entrada controlada para observar reconhecimento óptico de caracteres. | Notebook 01. |
| `brutos/pagina_digitalizada_degradada.png` | Versão da mesma imagem com redução, inclinação, desfoque e marcas. | Efeito da qualidade material e da digitalização sobre os erros de OCR. | Notebook 01. |
| `brutos/ocr_referencia.txt` | Transcrição correta usada como referência de controle. | Avaliação do OCR por comparação com uma transcrição validada; não é uma saída automática. | Notebook 01. |
| `brutos/ocr_precomputado.txt` | Saída de OCR previamente calculada para a imagem limpa. | Permitir o experimento offline e medir erros mesmo sem Tesseract instalado. | Notebook 01. |
| `brutos/ocr_precomputado_degradado.txt` | Saída previamente calculada para a imagem degradada, contendo erros deliberados. | Comparar CER e WER e mostrar que OCR produz hipótese textual, não transcrição garantida. | Notebook 01. |
| `brutos/indicadores_largos.csv` | Tabela fictícia com tema e período codificados nos nomes das colunas. | Diferença entre formatos largo e longo e explicitação das dimensões tema e período. | Notebooks 02 e 03. |
| `brutos/extrato_codigos_municipios_ibge.csv` | Extrato didático de quatro códigos municipais de uma fonte pública real. | Enriquecimento por junção, compatibilidade de chaves e cardinalidade muitos-para-um. | Notebooks 01 e 03. |
| `brutos/proveniencia_base_publica.json` | Fonte, URL, data de acesso e ressalva de reutilização do extrato do IBGE. | Proveniência e responsabilidade no uso de uma base pública; o CSV não documenta sozinho sua origem. | Notebook 01 e rastreamento do pacote final. |

Os arquivos de OCR pré-computado permanecem nessa camada porque funcionam
como entradas preservadas para o experimento reproduzível. Eles não devem ser
confundidos nem com as imagens que deram origem ao reconhecimento nem com a
transcrição de referência usada para avaliá-lo.

### Resultado verificável: `intermediarios/`

| Arquivo | O que representa | Problema ao qual se relaciona | Uso principal |
|---|---|---|---|
| `intermediarios/catalogo_normalizado.csv` | Versão do catálogo após regras explícitas de normalização, representação de datas incertas e sinalização de possíveis duplicatas. | Preservar decisões e incertezas entre a entrada bruta e a integração, sem sobrescrever o original. | Produzido no Notebook 02 e consumido no Notebook 03. |

### Produtos para análise: `derivados/`

| Arquivo | O que representa | Problema ao qual se relaciona | Uso principal |
|---|---|---|---|
| `derivados/documentos_processaveis.csv` | Tabela principal já normalizada e enriquecida com município e, quando disponível, texto. | Integrar entidades sem multiplicar documentos e preparar uma unidade documental analisável. | Produzido no Notebook 03 e auditado na oficina. |
| `derivados/documentos_temas.csv` | Tabela de relações, com uma linha para cada associação entre documento e tema. | Representar cardinalidade 1:N sem armazenar vários temas em uma única célula. | Produzido no Notebook 03 e auditado na oficina. |
| `derivados/indicadores_longos.csv` | Indicadores reorganizados com tema e período em colunas próprias. | Produzir estrutura longa adequada a agrupamentos e análises posteriores. | Produzido no Notebook 03 e auditado na oficina. |

Em síntese, o percurso é **fontes preservadas → transformação verificável →
produtos analisáveis**. Os notebooks devem reconstruir as duas últimas
camadas; por isso, não edite manualmente os arquivos brutos, intermediários
ou derivados.

Com esse mapa, podemos inspecionar o catálogo bruto e contar as entradas sem
ainda corrigi-las. A finalidade da primeira leitura é reconhecer o material,
não começar silenciosamente a limpeza.

In [2]:
from pathlib import Path
import pandas as pd

raiz_dados = Path("dados")
bruto = pd.read_csv(raiz_dados / "brutos" / "catalogo_messy.csv", sep=";")
print("Registros brutos:", len(bruto))
print("Arquivos brutos:", len(list((raiz_dados / "brutos").iterdir())))
bruto.head(3)

Registros brutos: 8
Arquivos brutos: 15


,id_documento,titulo,data_documento,municipio,codigo_municipio,genero,palavras
0,D001,Jornal Aurora,1890-01-05,São Paulo,3550308,Editorial,860.0
1,D002,gazeta popular,06/02/1891,Sao paulo,3550308,notícia,610.0
2,D003,Correio do Vale,1892,Belo Horizonte,3106200,Carta,NaN


A inspeção apresentou os materiais que atravessarão o percurso. Antes de
transformá-los, precisamos estabelecer onde cada estado dos dados ficará e
como será possível retornar ao ponto de entrada.

## Regra de ouro: camadas, não sobrescrita

- `brutos/`: cópias recebidas, imutáveis;
- `intermediarios/`: resultados de etapas verificáveis;
- `derivados/`: produtos destinados à análise;
- notebooks: ordem e justificativa das operações.

Uma cópia bruta não é “verdade original”; ela é o ponto de entrada preservado
do fluxo. Proveniência continua necessária.

Separar as camadas protege os dados recebidos, mas não identifica quais
operações serão necessárias em cada projeto. O diagnóstico antecipa essas
operações e os riscos que deverão ser registrados ao longo da unidade.

## U03-A01 — Diagnóstico

Que transformações seu projeto exigirá? Quais podem alterar sentido, unidade
ou quantidade de registros?

**Resposta:**

| Transformação prevista | Pode alterar | Risco principal |
|---|---|---|
| Ler o CSV com a codificação correta (UTF-8 com BOM) | Sentido | Uma codificação errada corrompe títulos e nomes acentuados ou em outros alfabetos |
| Padronizar o local de nascimento (cidades e estados dos EUA, nações do Reino Unido) | Sentido | Agregar escalas diferentes e apagar entidades históricas (URSS, Império Russo) |
| Sinalizar anos negativos, páginas suspeitas e contagens incoerentes | Nada, se apenas sinalizar | A tentação de “corrigir” e inventar valores |
| Criar chave de título para detectar duplicatas | Quantidade, se usada para excluir | Uma regra mal desenhada cria falsas duplicatas e apaga obras distintas |
| Auditar o gênero da autoria com o prenome | Sentido e quantidade nos grupos | Transformar uma inferência por nome em identidade de pessoas reais |
| Separar livros, autores e relações livro–gênero | Unidade | Uma junção mal declarada multiplica livros ou cria casos |
| Passar os gêneros literários para o formato longo | Unidade (de livro para relação livro–gênero) | Contar relações como se fossem livros |
| Calcular a taxa de resenhas | Sentido | Ler uma razão como medida de qualidade |

As transformações que podem mudar a quantidade de registros são as junções e
qualquer exclusão. Por isso, nenhuma exclusão será feita na base principal, e
cada junção terá cardinalidade declarada e testada.

As respostas do diagnóstico indicam transformações e riscos previstos. O
produto reúne as evidências necessárias para mostrar que essas operações
foram executadas, testadas e documentadas de maneira auditável.

## Produto e avaliação

A entrega incluirá inventário, código executável, tabela processável, log de
transformações, testes de chaves/junções, amostra de controle de OCR e limites.
Frequências substantivas, gráficos e hipóteses exploratórias pertencem à
Unidade 4. Consulte `referencias.md` para leituras completas. Comece pelo
Notebook 01, no qual os formatos recebidos determinarão leitores, parâmetros
e estratégias de extração diferentes.